In [ ]:
!pip -q install transformers datasets trl bitsandbytes accelerate peft

In [ ]:
from transformers import AutoTokenizer, TrainingArguments, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from getpass import getpass
from huggingface_hub import login
import torch
import os
import subprocess
from typing import Optional
from tqdm import tqdm
from textwrap import shorten
import gc
import random

In [ ]:
login(getpass("Input your HF Token Here: "))

Input your HF Token Here: ··········


In [ ]:
def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    la, lb = len(a), len(b)
    if la == 0:
        return lb
    if lb == 0:
        return la
    if la < lb:
        a, b = b, a
        la, lb = lb, la
    prev = list(range(lb + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            ins = prev[j] + 1
            delete = curr[j - 1] + 1
            replace = prev[j - 1] + (0 if ca == cb else 1)
            curr.append(min(ins, delete, replace))
        prev = curr
    return prev[-1]

def longestCommonSubsequence(text1, text2):
    m, n = len(text1), len(text2)
    dp_row1 = [0] * (n + 1)
    dp_row2 = [0] * (n + 1)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i - 1] == text2[j - 1]:
                dp_row2[j] = 1 + dp_row1[j - 1]
            else:
                dp_row2[j] = max(dp_row2[j - 1], dp_row1[j])
        dp_row1 = dp_row2
        dp_row2 = [0] * (n + 1)
    return dp_row1[-1]

def extract_md_code_block(st):
    if "```" not in st:
        return st
    else:
        return st[st.index("```") + 3:].replace("```", "")

def can_compile_assembly(asm_code: str, arch: str = "x86_64", *, verbose: bool = False) -> bool:
    arch_flag = "-m64" if arch.lower() in ("x86_64", "amd64") else "-m32"
    devnull = "/dev/null" if os.name != "nt" else "NUL"
    cmd = ["gcc", arch_flag, "-x", "assembler", "-c", "-", "-o", devnull]
    try:
        result = subprocess.run(cmd, input=asm_code.encode("utf-8"), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False)
    except FileNotFoundError:
        if verbose:
            print("Error: gcc not found in PATH")
        return False
    if result.returncode != 0:
        if verbose:
            print("Assembler diagnostics:")
            print(result.stderr.decode("utf-8", errors="replace"))
        return False
    return True

In [ ]:
model_name = "codellama/CodeLlama-7b-Instruct-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
dataset_name = "AzizZeynalli/c2x86-dataset"

dataset = load_dataset(dataset_name)
EOS_TOKEN = tokenizer.eos_token
tokenizer.pad_token = EOS_TOKEN

training_size = 5000
ds_shuffled = dataset['train'].shuffle(42).select(range(training_size))

validation_size = 200
validation_data = dataset['validation'].shuffle(42).select(range(validation_size))

In [ ]:
example_pool_data = dataset['train'].select(range(5000, 8000))

def get_random_examples(n=3):
    indices = random.sample(range(len(example_pool_data)), n)
    examples = []
    for idx in indices:
        examples.append({
            'c': example_pool_data[idx]['c'],
            's': example_pool_data[idx]['s']
        })
    return examples

In [ ]:
def create_fewshot_chat(input_c_code, num_examples=3):
    examples = get_random_examples(num_examples)

    chat = [{"role": "system", "content": "You are a compiler assistant who is able to compile C code into x86 assembly code in ATT syntax. You are working on Debian OS and you don't need to generate non-essential directives. Please provide your generated code directly without any prefix."}]

    for ex in examples:
        chat.append({"role": "user", "content": f"Please compile the following code \n {ex['c']}"})
        chat.append({"role": "assistant", "content": f"```{ex['s']}"})

    chat.append({"role": "user", "content": f"Please compile the following code \n {input_c_code}"})

    return chat

In [ ]:
eval_size = min(50, len(validation_data))
eval_ds = validation_data.select(range(eval_size))

results_baseline_metrics = []
for i, row in tqdm(enumerate(eval_ds)):
    prompt_chat = [
        {"role": "system", "content": "You are a compiler assistant who is able to compile C code into x86 assembly code in ATT syntax. You are working on Debian OS and you don't need to generate non-essential directives."},
        {"role": "user", "content": f"Please compile the following code \n {row['c']}"}
    ]
    prompt_text = tokenizer.apply_chat_template(prompt_chat, tokenize=False)

    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=4096)
    device_eval = next(model.parameters()).device
    inputs = {k: v.to(device_eval) for k, v in inputs.items()}

    try:
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False, eos_token_id=tokenizer.eos_token_id)
    except Exception as e:
        print(f"Generation error at sample {i}: {e}")
        break

    gen_text = tokenizer.decode(out[0], skip_special_tokens=True)
    if gen_text.startswith(prompt_text):
        pred = gen_text[len(prompt_text):].strip()
    else:
        pred = gen_text.replace(prompt_text, "").strip()

    end_word = "[/INST]"
    if end_word in pred:
        generation_idx = pred.rindex(end_word) + len(end_word)
        pred = pred[generation_idx:]

    pred = extract_md_code_block(pred)
    target = row['s'].strip()

    leve_dist = levenshtein(pred, target)
    lcs_dist = longestCommonSubsequence(pred, target)
    leve_norm = leve_dist / max(1, len(target))
    lcs_norm = lcs_dist / max(1, len(target))
    compilation_success = can_compile_assembly(pred)
    results_baseline_metrics.append({"pred": pred, "target": target, "leve_dist": leve_dist, "leve_norm": leve_norm, "lcs_dist": lcs_dist, "lcs_norm": lcs_norm, "compilation success": compilation_success})

0it [00:00, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
1it [00:15, 15.82s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
2it [00:30, 15.27s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
3it [00:45, 15.03s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
4it [01:00, 15.08s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
5it [01:15, 14.95s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
6it [01:30, 14.89s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
7it [01:45, 14.97s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
8it [01:59, 14.88s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
9it [02:14, 14.82s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
10it [02:29, 14.91s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
11it [02:52, 

In [ ]:
import numpy as np

In [ ]:
if len(results_baseline_metrics) > 0:
    lev_norms = [r['leve_norm'] for r in results_baseline_metrics]
    lcs_norms = [r['lcs_norm'] for r in results_baseline_metrics]
    compile_results = [r['compilation success'] for r in results_baseline_metrics]

    print(f"BASELINE | Samples: {len(results_baseline_metrics)}")
    print(f"Norm Lev - Mean: {np.mean(lev_norms):.3f}, Median: {np.median(lev_norms):.3f}, Std: {np.std(lev_norms):.3f}")
    print(f"Norm LCS - Mean: {np.mean(lcs_norms):.3f}, Median: {np.median(lcs_norms):.3f}, Std: {np.std(lcs_norms):.3f}")
    print(f"Compile Rate: {np.mean(compile_results):.3f}")

    for j, r in enumerate(results_baseline_metrics[:3]):
        print(f'\n--- Example {j} ---')
        print('Target:\n', r['target'][:300])
        print('Predicted:\n', r['pred'][:300])
        print(f"Norm Lev: {r['leve_norm']:.3f} | Norm LCS: {r['lcs_norm']:.3f} | Compiles: {r['compilation success']}")

BASELINE | Samples: 50
Norm Lev - Mean: 0.901, Median: 0.872, Std: 0.250
Norm LCS - Mean: 0.189, Median: 0.164, Std: 0.125
Compile Rate: 0.020

--- Example 0 ---
Target:
 .text
	.globl	gcd
gcd:
	pushq	%rbp
	movq	%rsp, %rbp
	movq	%rdi, -24(%rbp)
	movq	%rsi, -32(%rbp)
	cmpq	$0, -24(%rbp)
	jne	.L2
	movq	-32(%rbp), %rax
	jmp	.L3
.L2:
	cmpq	$0, -32(%rbp)
	jne	.L4
	movq	-24(%rbp), %rax
	jmp	.L3
.L4:
	movq	$0, -8(%rbp)
	jmp	.L5
.L6:
	sarq	-24(%rbp)
	sarq	-32(%rbp)
	addq	$1,
Predicted:
 
    .file   "fraction.c"
    .text
    .globl  gcd
    .type   gcd, @function
gcd:
.LFB0:
    pushq   %rbp
    movq    %rsp, %rbp
    movq    %rdi, -8(%rbp)
    movq    %rsi, -16(%rbp)
    movq    $0, -24(%rbp)
    movq    $0, -32(%rbp)
    movq    $0, -40(%rbp)
    movq    $0, -48(%rbp)
    movq  
Norm Lev: 0.936 | Norm LCS: 0.064 | Compiles: False

--- Example 1 ---
Target:
 .text
	.globl	stack_blinky1
	.bss
	.align 32
stack_blinky1:
	.zero	160
	.globl	blinky1
	.align 8
blinky1:
	.zero	8
	.text
	.globl	main_

In [ ]:
def df_row_to_chat_fewshot(df_row):
    examples = get_random_examples(3)

    chat_dict = [{"role": "system", "content": "You are a compiler assistant who is able to compile C code into x86 assembly code in ATT syntax. You are working on Debian OS and you don't need to generate non-essential directives. Please provide your generated code directly without any prefix."}]

    for ex in examples:
        chat_dict.append({"role": "user", "content": f"Please compile the following code \n {ex['c']}"})
        chat_dict.append({"role": "assistant", "content": f"```{ex['s']}"})

    chat_dict.append({"role": "user", "content": f"Please compile the following code \n {df_row['c']}"})
    chat_dict.append({"role": "assistant", "content": f"```{df_row['s']}"})

    return {'chat': chat_dict}

def formatting_prompts_func(example):
    example['text'] = tokenizer.apply_chat_template(example['chat'], tokenize=False)
    return example

chat_dataset_fewshot = ds_shuffled.map(df_row_to_chat_fewshot)
chat_dataset_fewshot = chat_dataset_fewshot.map(formatting_prompts_func, remove_columns=['c', 's', 'filename', 'chat'])

validation_chat_dataset_fewshot = validation_data.map(df_row_to_chat_fewshot).map(formatting_prompts_func, remove_columns=['c', 's', 'filename', 'chat'])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 79,953,920 || all params: 6,818,500,608 || trainable%: 1.1726


In [ ]:
os.environ["WANDB_DISABLED"] = "true"

training_args = TrainingArguments(
    output_dir="sft-codellama-fewshot",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=10,
    fp16=True,
    remove_unused_columns=False,
    report_to=None,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

trainer_fewshot = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=chat_dataset_fewshot,
    eval_dataset=validation_chat_dataset_fewshot,
    peft_config=lora_config,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
trainer_fewshot.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,0.519600,0.323202,0.335841,81920.000000,0.908050
20,0.307500,0.289251,0.299424,163840.000000,0.916970
30,0.308700,0.268218,0.284614,245760.000000,0.920880
40,0.280400,0.255701,0.257863,327680.000000,0.923436
50,0.272800,0.247795,0.251003,409600.000000,0.925469
60,0.255400,0.241700,0.231274,491520.000000,0.927566
70,0.236400,0.235747,0.242804,573440.000000,0.929052
80,0.241100,0.230960,0.232632,655360.000000,0.930557
90,0.236100,0.227605,0.228906,737280.000000,0.931637
100,0.214800,0.225209,0.218236,819200.000000,0.932620


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: 

TrainOutput(global_step=625, training_loss=0.20252696504592896, metrics={'train_runtime': 4220.9893, 'train_samples_per_second': 1.185, 'train_steps_per_second': 0.148, 'total_flos': 2.0543579357184e+17, 'train_loss': 0.20252696504592896, 'entropy': 0.15866020284593105, 'num_tokens': 5120000.0, 'mean_token_accuracy': 0.9498289346694946, 'epoch': 1.0})

In [ ]:
trainer_fewshot.save_model("codellama-peft-sft-c2x86-fewshot")
model.push_to_hub("AzizZeynalli/codellama-4bit-peft-lora-r32-sft-c2x86-fewshot")
tokenizer.push_to_hub("AzizZeynalli/codellama-4bit-peft-lora-r32-sft-c2x86-fewshot")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  558kB /  320MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...p49n_gz_3/tokenizer.model:  95%|#########4|  473kB /  500kB            

CommitInfo(commit_url='https://huggingface.co/AzizZeynalli/codellama-4bit-peft-lora-r32-sft-c2x86-fewshot/commit/5c9df7a592516fe40fea035086b510c300ba5b06', commit_message='Upload tokenizer', commit_description='', oid='5c9df7a592516fe40fea035086b510c300ba5b06', pr_url=None, repo_url=RepoUrl('https://huggingface.co/AzizZeynalli/codellama-4bit-peft-lora-r32-sft-c2x86-fewshot', endpoint='https://huggingface.co', repo_type='model', repo_id='AzizZeynalli/codellama-4bit-peft-lora-r32-sft-c2x86-fewshot'), pr_revision=None, pr_num=None)

In [ ]:
n_examples = 5
sample = validation_data.select(range(min(n_examples, len(validation_data))))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.eval()

results_after_zeroshot = []
for i, row in enumerate(sample):
    prompt_chat = [
        {"role": "system", "content": "You are a compiler assistant who is able to compile C code into x86 assembly code in ATT syntax. You are working on Debian OS and you don't need to generate non-essential directives. Please provide your generated code directly without any prefix."},
        {"role": "user", "content": f"Please compile the following code \n {row['c']}"}
    ]
    prompt_text = tokenizer.apply_chat_template(prompt_chat, tokenize=False)
    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=4096)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False, eos_token_id=tokenizer.eos_token_id)
    gen_text = tokenizer.decode(out[0], skip_special_tokens=True)
    if gen_text.startswith(prompt_text):
        pred_after = gen_text[len(prompt_text):].strip()
    else:
        pred_after = gen_text.replace(prompt_text, "").strip()

    results_after_zeroshot.append({"c": row['c'], "s": row['s'].strip(), "after": pred_after})

print('\nAfter fine-tune with ZERO-shot inference:')
for i, r in enumerate(results_after_zeroshot):
    print(f'\n=== Example {i} ===')
    print('\n--- Input C code ---\n', shorten(r['c'], width=400, placeholder='...'))
    print('\n--- Target ---\n', shorten(r['s'], width=500, placeholder='...'))
    end_word = "[/INST]"
    if end_word in r['after']:
        generation_idx = r['after'].rindex(end_word) + len(end_word)
        generated = extract_md_code_block(r['after'][generation_idx:])
        print('\n--- After fine-tune ---\n', shorten(generated, width=500, placeholder='...'))
    else:
        generated = extract_md_code_block(r['after'])
        print('\n--- After fine-tune ---\n', shorten(generated, width=500, placeholder='...'))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



After fine-tune with ZERO-shot inference:

=== Example 0 ===

--- Input C code ---
 #include <stdio.h> long long gcd(long long u, long long v) { long long shift; if (u == 0L) return v; if (v == 0L) return u; for (shift = 0L; ((u | v) & 1L) == 0L; ++shift) { u >>= 1; v >>= 1; } while ((u & 1L) == 0L) u >>= 1L; do { while ((v & 1L) == 0L) v >>= 1L; if (u > v) { long long t = v; v = u; u = t; } v = v - u; } while (v != 0L); return u << shift; } struct fraction { long long a, b;...

--- Target ---
 .text .globl gcd gcd: pushq %rbp movq %rsp, %rbp movq %rdi, -24(%rbp) movq %rsi, -32(%rbp) cmpq $0, -24(%rbp) jne .L2 movq -32(%rbp), %rax jmp .L3 .L2: cmpq $0, -32(%rbp) jne .L4 movq -24(%rbp), %rax jmp .L3 .L4: movq $0, -8(%rbp) jmp .L5 .L6: sarq -24(%rbp) sarq -32(%rbp) addq $1, -8(%rbp) .L5: movq -24(%rbp), %rax orq -32(%rbp), %rax andl $1, %eax testq %rax, %rax je .L6 jmp .L7 .L8: sarq -24(%rbp) .L7: movq -24(%rbp), %rax andl $1, %eax testq %rax, %rax je .L8 jmp .L9 .L10: sarq -32(%rbp)...

In [ ]:
from peft import PeftModel

model_name = "codellama/CodeLlama-7b-Instruct-hf"
adapter_name = "AzizZeynalli/codellama-4bit-peft-lora-r32-sft-c2x86-fewshot"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, adapter_name)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/320M [00:00<?, ?B/s]

In [ ]:
eval_size = min(50, len(validation_data))
eval_ds = validation_data.select(range(eval_size))

results_after_zeroshot_metrics = []
for i, row in tqdm(enumerate(eval_ds)):
    prompt_chat = [
        {"role": "system", "content": "You are a compiler assistant who is able to compile C code into x86 assembly code in ATT syntax. You are working on Debian OS and you don't need to generate non-essential directives."},
        {"role": "user", "content": f"Please compile the following code \n {row['c']}"}
    ]
    prompt_text = tokenizer.apply_chat_template(prompt_chat, tokenize=False)

    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=4096)
    device_eval = next(model.parameters()).device
    inputs = {k: v.to(device_eval) for k, v in inputs.items()}

    try:
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False, eos_token_id=tokenizer.eos_token_id)
    except Exception as e:
        print(f"Generation error at sample {i}: {e}")
        break

    gen_text = tokenizer.decode(out[0], skip_special_tokens=True)
    if gen_text.startswith(prompt_text):
        pred = gen_text[len(prompt_text):].strip()
    else:
        pred = gen_text.replace(prompt_text, "").strip()

    end_word = "[/INST]"
    if end_word in pred:
        generation_idx = pred.rindex(end_word) + len(end_word)
        pred = pred[generation_idx:]

    pred = extract_md_code_block(pred)
    target = row['s'].strip()

    leve_dist = levenshtein(pred, target)
    lcs_dist = longestCommonSubsequence(pred, target)
    leve_norm = leve_dist / max(1, len(target))
    lcs_norm = lcs_dist / max(1, len(target))
    compilation_success = can_compile_assembly(pred)
    results_after_zeroshot_metrics.append({"pred": pred, "target": target, "leve_dist": leve_dist, "leve_norm": leve_norm, "lcs_dist": lcs_dist, "lcs_norm": lcs_norm, "compilation success": compilation_success})

0it [00:00, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
1it [00:29, 29.94s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
2it [00:59, 29.55s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
3it [01:28, 29.24s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
4it [01:57, 29.34s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
5it [02:26, 29.18s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
6it [02:55, 29.15s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
7it [03:24, 29.16s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
8it [03:53, 29.12s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
9it [04:22, 29.06s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
10it [04:52, 29.20s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
11it [05:24, 

In [ ]:
if len(results_after_zeroshot_metrics) > 0:
    avg_leve_dist = sum(r['leve_dist'] for r in results_after_zeroshot_metrics) / len(results_after_zeroshot_metrics)
    avg_leve_norm = sum(r['leve_norm'] for r in results_after_zeroshot_metrics) / len(results_after_zeroshot_metrics)
    avg_lcs_dist = sum(r['lcs_dist'] for r in results_after_zeroshot_metrics) / len(results_after_zeroshot_metrics)
    avg_lcs_norm = sum(r['lcs_norm'] for r in results_after_zeroshot_metrics) / len(results_after_zeroshot_metrics)
    avg_compilation_success = sum(r['compilation success'] for r in results_after_zeroshot_metrics) / len(results_after_zeroshot_metrics)
    print(f"POST-TRAINING ZERO-SHOT | Samples: {len(results_after_zeroshot_metrics)} | Lev Dist: {avg_leve_dist:.2f} | Norm Lev: {avg_leve_norm:.3f} | LCS: {avg_lcs_dist:.3f} | Norm LCS: {avg_lcs_norm:.3f} | Compile Rate: {avg_compilation_success:.3f}")

    for j, r in enumerate(results_after_zeroshot_metrics[:3]):
        print(f'\n--- Example {j} ---')
        print('Target:\n', r['target'][:300])
        print('Predicted:\n', r['pred'][:300])
        print(f"Norm Lev: {r['leve_norm']:.3f} | Norm LCS: {r['lcs_norm']:.3f} | Compiles: {r['compilation success']}")

POST-TRAINING ZERO-SHOT | Samples: 50 | Lev Dist: 2864.56 | Norm Lev: 0.740 | LCS: 375.040 | Norm LCS: 0.267 | Compile Rate: 0.300

--- Example 0 ---
Target:
 .text
	.globl	gcd
gcd:
	pushq	%rbp
	movq	%rsp, %rbp
	movq	%rdi, -24(%rbp)
	movq	%rsi, -32(%rbp)
	cmpq	$0, -24(%rbp)
	jne	.L2
	movq	-32(%rbp), %rax
	jmp	.L3
.L2:
	cmpq	$0, -32(%rbp)
	jne	.L4
	movq	-24(%rbp), %rax
	jmp	.L3
.L4:
	movq	$0, -8(%rbp)
	jmp	.L5
.L6:
	sarq	-24(%rbp)
	sarq	-32(%rbp)
	addq	$1,
Predicted:
 	.text
	.globl	gcd
gcd:
	pushq	%rbp
	movq	%rsp, %rbp
	movq	%rdi, -24(%rbp)
	movq	%rsi, -32(%rbp)
	cmpq	$0, -24(%rbp)
	jne	.L2
	movq	-32(%rbp), %rax
	jmp	.L3
.L2:
	cmpq	$0, -32(%rbp)
	jne	.L4
	movq	-24(%rbp), %rax
	jmp	.L3
.L4:
	movq	$0, -8(%rbp)
	jmp	.L5
.L8:
	movq	-24(%rbp), %rax
	sarq	%rax
	movq	%
Norm Lev: 0.937 | Norm LCS: 0.063 | Compiles: False

--- Example 1 ---
Target:
 .text
	.globl	stack_blinky1
	.bss
	.align 32
stack_blinky1:
	.zero	160
	.globl	blinky1
	.align 8
blinky1:
	.zero	8
	.text
	.globl	main_blinky1
main